# Стійкість згорткової нейромережі до змагальних атак FGSM і PGD та ефективність змагального навчання

**Задача.** Класифікація жестів «камінь — ножиці — папір» (Rock-Paper-Scissors) згортковою нейромережею (CNN) і тестування її на проникнення (ML pentest) змагальними атаками у режимі *white-box*.

**Що зроблено в ноутбуці:**

1. Розвідувальний аналіз набору даних (EDA).
2. Навчання базової CNN і оцінка її якості на чистих даних.
3. Атака **FGSM** (Fast Gradient Sign Method) — одноетапна градієнтна атака.
4. Атака **PGD** (Projected Gradient Descent) — ітеративна, сильніша версія FGSM.
5. Візуалізація змагальних прикладів.
6. Захист: **змагальне навчання** (adversarial training) на прикладах FGSM.
7. Порівняння стійкості моделей до та після захисту і висновки для публікації.

**Головна метрика** — точність (accuracy) на тестовій вибірці залежно від бюджету збурення ε, тобто максимально дозволеної зміни кожного пікселя (у шкалі [0, 1]). Для орієнтиру: ε = 8/255 ≈ 0.031 — стандартний бюджет у літературі, за якого збурення зазвичай непомітне для людини.

> Усі рисунки автоматично зберігаються у папку `figures/` у роздільній здатності 300 dpi — їх можна напряму вставляти в статтю.

## 0. Налаштування середовища

Встановлюємо залежності, фіксуємо зерна генераторів випадкових чисел (для відтворюваності результатів) і задаємо єдиний стиль графіків.

In [ ]:
# kagglehub потрібен для завантаження набору даних з Kaggle
!pip install -q kagglehub

In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

from sklearn.metrics import confusion_matrix, classification_report
from IPython.display import display, Markdown

# ---------------------------------------------------------------
# Відтворюваність: фіксуємо всі джерела випадковості
# ---------------------------------------------------------------
SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()

# Якщо доступна GPU — рахуємо на ній, інакше на CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ---------------------------------------------------------------
# Єдиний стиль графіків для всієї роботи
# ---------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,            # якість для публікації
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#8a8984",
    "axes.axisbelow": True,        # сітка під даними
    "axes.grid": True,
    "grid.color": "#d9d8d4",
    "grid.linewidth": 0.8,
    "grid.alpha": 0.7,
    "legend.frameon": False,
    "lines.linewidth": 2,
    "lines.markersize": 5,
})

# Кольори несуть сенс і однакові на всіх рисунках:
#   синій   — стандартна модель, помаранчевий — модель після змагального навчання
#   тип атаки кодується стилем лінії/штрихуванням, а не кольором
MODEL_COLORS = {"Стандартна": "#2a78d6", "Змагальне навчання": "#eb6834"}
ATTACK_STYLE = {
    "FGSM": dict(linestyle="-", marker="o"),
    "PGD": dict(linestyle="--", marker="s"),
}
ATTACK_HATCH = {"FGSM": "", "PGD": "////"}
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]   # paper, rock, scissors
INK = "#0b0b0b"
INK_MUTED = "#52514e"

# Послідовна одно-тонова шкала (світлий -> темний синій) для теплових карт
SEQ_CMAP = LinearSegmentedColormap.from_list(
    "seq_blue",
    ["#ffffff", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"],
)

# Усі рисунки дублюємо у файли для статті
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)


def save_fig(fig, name):
    """Зберігає рисунок у figures/<name>.png (300 dpi)."""
    fig.savefig(FIG_DIR / f"{name}.png")


def pct(x):
    """0.9658 -> '96.6%'."""
    return f"{100 * x:.1f}%"

## 1. Набір даних і розвідувальний аналіз (EDA)

Використовуємо відкритий набір [Rock-Paper-Scissors](https://www.kaggle.com/datasets/drgfreeman/rockpaperscissors) (Kaggle, `drgfreeman/rockpaperscissors`): фотографії руки, що показує один із трьох жестів, на однорідному зеленому фоні.

Перед навчанням перевіряємо: скільки зображень у кожному класі (чи збалансований набір), які розміри та колірні режими зображень, чи немає пошкоджених файлів.

In [ ]:
import kagglehub

# Завантаження набору (у Colab/Kaggle береться з кешу)
path = kagglehub.dataset_download("drgfreeman/rockpaperscissors")

# Зображення лежать у підпапці rps-cv-images/<клас>/*.png
DATA_ROOT = Path(path) / "rps-cv-images"

print(path)
print(os.listdir(path))

In [ ]:
# ---------------------------------------------------------------
# Збираємо метадані кожного зображення в таблицю
# ---------------------------------------------------------------
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

rows = []
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not class_dir.is_dir():
        continue

    for image_path in class_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        try:
            with Image.open(image_path) as img:
                arr = np.asarray(img)
                rows.append({
                    "class": class_dir.name,
                    "file": image_path.name,
                    "path": str(image_path),
                    "width": img.size[0],
                    "height": img.size[1],
                    "mode": img.mode,
                    "mean_pixel": arr.mean(),   # середня яскравість
                    "std_pixel": arr.std(),     # контраст
                })
        except Exception as e:          # пошкоджений файл — повідомляємо і пропускаємо
            print(f"Could not read {image_path}: {e}")

eda_df = pd.DataFrame(rows)

class_counts = eda_df["class"].value_counts().sort_index()
resolutions = Counter(zip(eda_df["width"], eda_df["height"]))

print(f"Всього зображень: {len(eda_df)}")
print(f"Кількість класів: {eda_df['class'].nunique()}  {sorted(eda_df['class'].unique())}")
print(f"Колірні режими: {dict(eda_df['mode'].value_counts())}")
print("Роздільна здатність: " + ", ".join(f"{w}x{h} ({n})" for (w, h), n in resolutions.most_common()))

# Зведення по класах
summary = eda_df.groupby("class").agg(
    images=("file", "count"),
    share=("file", lambda s: len(s) / len(eda_df)),
    avg_pixel_mean=("mean_pixel", "mean"),
    avg_pixel_std=("std_pixel", "mean"),
)
display(summary.style.format({"share": "{:.1%}", "avg_pixel_mean": "{:.1f}", "avg_pixel_std": "{:.1f}"}))

In [ ]:
# ---------------------------------------------------------------
# Рис. 1. Розподіл класів + розподіл яскравості зображень
# (стовпчикова діаграма читається точніше за кругову)
# ---------------------------------------------------------------
classes_sorted = list(class_counts.index)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 1.4]})

bars = ax1.barh(classes_sorted, class_counts.values, color=CLASS_COLORS[:len(classes_sorted)], height=0.6)
for bar, n in zip(bars, class_counts.values):
    ax1.text(bar.get_width() + class_counts.max() * 0.02, bar.get_y() + bar.get_height() / 2,
             f"{n} ({n / len(eda_df):.1%})", va="center", color=INK, fontsize=10)
ax1.set_xlim(0, class_counts.max() * 1.3)
ax1.invert_yaxis()
ax1.grid(axis="y", visible=False)
ax1.set_xlabel("Кількість зображень")
ax1.set_title("Розподіл класів")

bins = np.linspace(eda_df["mean_pixel"].min(), eda_df["mean_pixel"].max(), 35)
for color, cls in zip(CLASS_COLORS, classes_sorted):
    ax2.hist(eda_df.loc[eda_df["class"] == cls, "mean_pixel"], bins=bins,
             histtype="step", linewidth=2, color=color, label=cls)
ax2.set_xlabel("Середня яскравість зображення (0–255)")
ax2.set_ylabel("Кількість зображень")
ax2.set_title("Розподіл яскравості за класами")
ax2.legend()

fig.tight_layout()
save_fig(fig, "fig01_class_distribution")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Рис. 2. Випадкові приклади кожного класу
# ---------------------------------------------------------------
SAMPLES_PER_CLASS = 4

fig, axes = plt.subplots(len(classes_sorted), SAMPLES_PER_CLASS,
                         figsize=(2.6 * SAMPLES_PER_CLASS, 1.9 * len(classes_sorted)),
                         squeeze=False)

for i, cls in enumerate(classes_sorted):
    sample = eda_df[eda_df["class"] == cls].sample(SAMPLES_PER_CLASS, random_state=SEED)
    for j, (_, row) in enumerate(sample.iterrows()):
        ax = axes[i][j]
        with Image.open(row["path"]) as img:
            ax.imshow(img)
        ax.set_xticks([]); ax.set_yticks([])
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[i][0].set_ylabel(cls, fontsize=12, rotation=0, ha="right", va="center", labelpad=10)

fig.suptitle("Приклади зображень набору Rock-Paper-Scissors", fontweight="bold")
fig.tight_layout()
save_fig(fig, "fig02_samples")
plt.show()

**Висновки з EDA**

* Набір містить **2188** RGB-зображень трьох класів; класи **майже збалансовані** (≈ 32.5 % / 33.2 % / 34.3 %), тому accuracy є коректною основною метрикою і додаткове зважування класів не потрібне.
* Усі зображення мають однаковий розмір **300×200** і однорідний зелений фон — умови зйомки контрольовані, задача для CNN відносно проста.
* Розподіли яскравості класів сильно перекриваються: модель не може «вгадати» клас за простою статистикою пікселів і змушена вивчати форму руки.

## 2. Підготовка даних

* Зменшуємо зображення до **64×64** (достатньо для розпізнавання жесту, навчання швидке навіть на CPU).
* Пікселі переводимо у діапазон **[0, 1]** без нормалізації — так бюджет атаки ε має прямий фізичний зміст: «максимальна зміна яскравості пікселя».
* Ділимо дані **80 % / 20 %** на навчальну та тестову вибірки з фіксованим зерном.

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                  # -> тензор [3, 64, 64] зі значеннями в [0, 1]
])

dataset = datasets.ImageFolder(root=DATA_ROOT, transform=transform)
class_names = dataset.classes

print(f"Класи: {dataset.class_to_idx}")
print(f"Всього зображень: {len(dataset)}")

In [ ]:
# Фіксований поділ 80/20 — однаковий при кожному запуску
generator = torch.Generator().manual_seed(SEED)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Швидка перевірка форми батчу
images, labels = next(iter(train_loader))
print(f"Train: {len(train_dataset)} | Test: {len(test_dataset)} | батчів у train: {len(train_loader)}")
print(f"Форма батчу зображень: {tuple(images.shape)}, міток: {tuple(labels.shape)}")

## 3. Архітектура моделі

Компактна CNN із двох згорткових блоків і двох повнозв'язних шарів:

| Шар | Вихідна форма |
|---|---|
| Вхід | 3 × 64 × 64 |
| Conv 3×3 (16) + ReLU + MaxPool 2 | 16 × 31 × 31 |
| Conv 3×3 (32) + ReLU + MaxPool 2 | 32 × 14 × 14 |
| Flatten → Linear (128) + ReLU | 128 |
| Linear (3) — логіти класів | 3 |

In [ ]:
class CNN(nn.Module):
    """Невелика згорткова мережа для класифікації 64x64 RGB-зображень на 3 класи."""

    def __init__(self, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)        # 64 -> 62
        self.conv2 = nn.Conv2d(16, 32, 3)       # 31 -> 29
        self.pool = nn.MaxPool2d(2)             # зменшує розмір удвічі
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # -> 16 x 31 x 31
        x = self.pool(self.relu(self.conv2(x)))  # -> 32 x 14 x 14
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)                       # логіти (softmax — усередині CrossEntropyLoss)


n_params = sum(p.numel() for p in CNN().parameters())
print(f"Кількість параметрів моделі: {n_params:,}")

## 4. Навчання базової (стандартної) моделі

Оптимізатор Adam (lr = 0.001), функція втрат — крос-ентропія, 10 епох. Після кожної епохи фіксуємо втрати та точність на навчальній і тестовій вибірках, щоб побачити динаміку навчання і можливе перенавчання.

In [ ]:
NUM_EPOCHS = 10
LR = 1e-3


def evaluate_clean(model, loader):
    """Точність моделі на чистих (не атакованих) даних."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total


set_seed()
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history_std = {"loss": [], "train_acc": [], "test_acc": []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    history_std["loss"].append(running_loss / len(train_loader))
    history_std["train_acc"].append(correct / total)
    history_std["test_acc"].append(evaluate_clean(model, test_loader))

    print(f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | loss {history_std['loss'][-1]:.4f} | "
          f"train acc {history_std['train_acc'][-1]:.4f} | test acc {history_std['test_acc'][-1]:.4f}")

test_accuracy = history_std["test_acc"][-1]
print(f"\nТочність на тестовій вибірці: {test_accuracy:.4f}")

In [ ]:
# ---------------------------------------------------------------
# Рис. 3. Криві навчання стандартної моделі
# (дві окремі панелі замість двох осей Y на одному графіку)
# ---------------------------------------------------------------
epochs = np.arange(1, NUM_EPOCHS + 1)
c = MODEL_COLORS["Стандартна"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history_std["loss"], color=c, marker="o")
ax1.set_title("Функція втрат (train)")
ax1.set_xlabel("Епоха"); ax1.set_ylabel("Cross-entropy")
ax1.set_ylim(bottom=0)

ax2.plot(epochs, history_std["train_acc"], color=c, marker="o", linestyle="--", label="Навчальна вибірка")
ax2.plot(epochs, history_std["test_acc"], color=c, marker="s", label="Тестова вибірка")
ax2.set_title("Точність")
ax2.set_xlabel("Епоха"); ax2.set_ylabel("Accuracy")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax2.legend(loc="lower right")

for ax in (ax1, ax2):
    ax.set_xticks(epochs)

fig.tight_layout()
save_fig(fig, "fig03_training_curves_standard")
plt.show()

## 5. Якість на чистих даних

Матриця помилок показує, які саме жести модель плутає. Значення нормовані на кількість прикладів справжнього класу (кожен рядок = 100 %), у дужках — абсолютна кількість.

In [ ]:
def predict_all(model, loader):
    """Повертає справжні мітки та передбачення для всієї вибірки."""
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(dim=1).cpu()
            y_true.extend(labels.numpy())
            y_pred.extend(preds.numpy())
    return np.array(y_true), np.array(y_pred)


def plot_confusion_matrix(y_true, y_pred, class_names, title, ax):
    """Нормована матриця помилок із підписами 'відсоток (кількість)'."""
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    ax.imshow(cm_norm, cmap=SEQ_CMAP, vmin=0, vmax=1)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f"{cm_norm[i, j]:.1%}\n({cm[i, j]})", ha="center", va="center",
                    fontsize=10, color="white" if cm_norm[i, j] > 0.55 else INK)
    ax.set_xticks(range(len(class_names)), class_names)
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("Передбачений клас")
    ax.set_ylabel("Справжній клас")
    ax.set_title(title)
    ax.grid(False)


y_true, y_pred = predict_all(model, test_loader)

fig, ax = plt.subplots(figsize=(5.2, 4.6))
plot_confusion_matrix(y_true, y_pred, class_names, f"Матриця помилок (accuracy {pct(test_accuracy)})", ax)
fig.tight_layout()
save_fig(fig, "fig04_confusion_matrix")
plt.show()

# Precision / recall / F1 для кожного класу
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
display(pd.DataFrame(report).T.round(3))

**Інтерпретація.** Базова модель досягає високої точності на чистих даних (≈ 96–97 % в авторському запуску), помилки поодинокі і розподілені між класами без явного перекосу. Отже, будь-яке подальше падіння точності пояснюється саме атакою, а не слабкістю моделі.

## 6. Атака FGSM (Fast Gradient Sign Method)

FGSM (Goodfellow et al., 2015) робить **один крок** у напрямку, який найшвидше збільшує функцію втрат:

$$x_{adv} = \mathrm{clip}_{[0,1]}\big(x + \varepsilon \cdot \mathrm{sign}(\nabla_x L(f(x), y))\big)$$

* $\varepsilon$ — бюджет збурення: кожен піксель змінюється не більше ніж на $\varepsilon$ (норма $L_\infty$);
* атака *white-box*: зловмисник знає модель і може обчислити її градієнт;
* атака *untargeted*: мета — будь-яка неправильна відповідь.

In [ ]:
def fgsm_attack(images, epsilon, gradients):
    """Один крок FGSM: зсув кожного пікселя на ±epsilon за знаком градієнта."""
    perturbation = epsilon * gradients.sign()
    adversarial_images = images + perturbation
    return torch.clamp(adversarial_images, 0, 1)       # лишаємося в допустимому діапазоні пікселів


def generate_fgsm(model, images, labels, criterion, epsilon):
    """Обчислює градієнт втрат за вхідним зображенням і будує FGSM-приклад."""
    images = images.clone().detach().requires_grad_(True)
    loss = criterion(model(images), labels)
    model.zero_grad()
    loss.backward()
    return fgsm_attack(images, epsilon, images.grad).detach()


def evaluate_under_attack(model, loader, attack_fn=None):
    """
    Точність моделі на змагальних прикладах.
    attack_fn(images, labels) -> adv_images; None означає чисті дані.
    """
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        adv_images = attack_fn(images, labels) if attack_fn is not None else images
        with torch.no_grad():
            predictions = model(adv_images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return correct / total


def eval_fgsm(model, loader, criterion, epsilon):
    return evaluate_under_attack(
        model, loader, lambda x, y: generate_fgsm(model, x, y, criterion, epsilon)
    )


# Перевірка коректності: при epsilon = 0 точність має збігатися з чистою
print(f"FGSM, eps=0: {eval_fgsm(model, test_loader, criterion, 0.0):.4f}  (clean: {test_accuracy:.4f})")

In [ ]:
# Сітка бюджетів збурення: дрібний крок там, де точність змінюється найшвидше
EPSILONS = [0.000, 0.0025, 0.005, 0.0075, 0.010, 0.0125, 0.015, 0.0175, 0.020,
            0.025, 0.030, 0.040, 0.050, 0.075, 0.100, 0.150, 0.200]


def run_sweep(eval_fn, epsilons, tag):
    """Проганяє атаку для кожного epsilon і повертає таблицю epsilon -> accuracy."""
    results = []
    for eps in epsilons:
        acc = eval_fn(eps)
        results.append({"epsilon": eps, "eps_255": eps * 255, "accuracy": acc})
        print(f"{tag} | eps = {eps:.4f} ({eps * 255:5.2f}/255) | accuracy = {acc:.4f}")
    return pd.DataFrame(results)


fgsm_results_df = run_sweep(lambda e: eval_fgsm(model, test_loader, criterion, e), EPSILONS, "FGSM")

## 7. Атака PGD (Projected Gradient Descent)

PGD (Madry et al., 2018) — ітеративна версія FGSM і де-факто стандарт для оцінки стійкості:

1. стартуємо з випадкової точки всередині $\varepsilon$-кулі навколо $x$;
2. робимо `steps` малих кроків розміру $\alpha$ за знаком градієнта;
3. після кожного кроку **проєктуємо** збурення назад у $[-\varepsilon, \varepsilon]$ і зображення — у $[0, 1]$.

**Вибір кроку $\alpha$.** Крок має масштабуватися разом з $\varepsilon$: за замовчуванням $\alpha = 2.5\,\varepsilon / \text{steps}$ (рекомендація Madry et al.). Фіксований малий крок (напр. $\alpha = 0.005$ при 10 кроках) не дозволяє атаці «дійти» до межі великого бюджету і **штучно занижує** силу PGD при $\varepsilon > 0.05$ — у первинній версії експерименту саме через це PGD на великих $\varepsilon$ виглядала слабшою за FGSM.

In [ ]:
PGD_STEPS = 10


def pgd_attack(model, images, labels, criterion, epsilon=0.03, alpha=None, steps=PGD_STEPS):
    """PGD з випадковим стартом у L-inf кулі радіуса epsilon."""
    if alpha is None:
        alpha = 2.5 * epsilon / steps                      # крок масштабується з бюджетом

    original = images.detach()
    adv = original + torch.empty_like(original).uniform_(-epsilon, epsilon)   # випадковий старт
    adv = torch.clamp(adv, 0, 1)

    for _ in range(steps):
        adv.requires_grad_(True)
        loss = criterion(model(adv), labels)
        model.zero_grad()
        loss.backward()

        adv = adv.detach() + alpha * adv.grad.sign()                       # крок підйому по втратах
        delta = torch.clamp(adv - original, -epsilon, epsilon)             # проєкція на eps-кулю
        adv = torch.clamp(original + delta, 0, 1)                          # проєкція на [0, 1]

    return adv.detach()


def evaluate_pgd(model, loader, criterion, epsilon=0.03, alpha=None, steps=PGD_STEPS):
    return evaluate_under_attack(
        model, loader,
        lambda x, y: pgd_attack(model, x, y, criterion, epsilon, alpha, steps),
    )


# Точкове порівняння на стандартному бюджеті eps = 0.03 (~ 8/255)
EPS_REF = 0.03
set_seed()
fgsm_ref = eval_fgsm(model, test_loader, criterion, EPS_REF)
pgd_ref = evaluate_pgd(model, test_loader, criterion, EPS_REF)

print(f"Clean accuracy:       {test_accuracy:.4f}")
print(f"FGSM  eps={EPS_REF}:     {fgsm_ref:.4f}")
print(f"PGD   eps={EPS_REF}:     {pgd_ref:.4f}")

In [ ]:
set_seed()
pgd_results_df = run_sweep(lambda e: evaluate_pgd(model, test_loader, criterion, e), EPSILONS, "PGD ")

## 8. Порівняння FGSM і PGD на стандартній моделі

Ключові величини для порівняння атак:

* **accuracy при ε = 0.03** (≈ 8/255) — стандартна точка порівняння в літературі;
* **ε₅₀** — бюджет, за якого точність падає нижче 50 % (чим менше, тим вразливіша модель);
* **AUC** — нормована площа під кривою accuracy(ε) на відрізку [0; 0.1] (інтегральна стійкість, 1 = ідеальна).

In [ ]:
def acc_at(df, eps):
    """Точність при заданому epsilon (лінійна інтерполяція між точками сітки)."""
    return float(np.interp(eps, df["epsilon"], df["accuracy"]))


def eps_at_threshold(df, threshold=0.5):
    """Найменший epsilon, за якого точність опускається нижче порогу (з інтерполяцією)."""
    e, a = df["epsilon"].to_numpy(), df["accuracy"].to_numpy()
    below = np.where(a < threshold)[0]
    if len(below) == 0:
        return np.nan                                  # поріг не досягнуто на сітці
    i = below[0]
    if i == 0:
        return e[0]
    return e[i - 1] + (a[i - 1] - threshold) / (a[i - 1] - a[i]) * (e[i] - e[i - 1])


def robustness_auc(df, eps_max=0.1):
    """Нормована площа під кривою accuracy(eps) на [0, eps_max] (метод трапецій)."""
    grid = np.linspace(0, eps_max, 201)
    acc = np.interp(grid, df["epsilon"], df["accuracy"])
    return float(np.sum((acc[1:] + acc[:-1]) / 2 * np.diff(grid)) / eps_max)


def summarize(curves):
    """curves: {(модель, атака): df} -> зведена таблиця метрик стійкості."""
    rows = []
    for (model_name, attack), df in curves.items():
        rows.append({
            "Модель": model_name,
            "Атака": attack,
            "Clean": acc_at(df, 0.0),
            "ε=0.01": acc_at(df, 0.01),
            "ε=0.03": acc_at(df, 0.03),
            "ε=0.05": acc_at(df, 0.05),
            "ε=0.10": acc_at(df, 0.10),
            "ε₅₀": eps_at_threshold(df),
            "AUC[0;0.1]": robustness_auc(df),
        })
    return pd.DataFrame(rows)


def style_summary(table):
    fmt = {c: "{:.1%}" for c in ["Clean", "ε=0.01", "ε=0.03", "ε=0.05", "ε=0.10"]}
    fmt.update({"ε₅₀": "{:.3f}", "AUC[0;0.1]": "{:.3f}"})
    return table.style.format(fmt, na_rep="> max ε").hide(axis="index")


def plot_robustness(ax, curves, eps_ref=EPS_REF, annotate=True):
    """Криві accuracy(eps): колір = модель, стиль лінії = атака."""
    for (model_name, attack), df in curves.items():
        ax.plot(df["epsilon"], df["accuracy"], color=MODEL_COLORS[model_name],
                label=f"{model_name} — {attack}", **ATTACK_STYLE[attack])
        if annotate:
            y = acc_at(df, eps_ref)
            dx, ha = (8, "left") if attack == "FGSM" else (-8, "right")
            ax.annotate(pct(y), (eps_ref, y), xytext=(dx, 6 if attack == "FGSM" else -12),
                        textcoords="offset points", ha=ha, fontsize=9, color=INK)

    ax.axvline(eps_ref, color=INK_MUTED, linestyle=":", linewidth=1.2)
    ax.text(eps_ref, 0.985, f" ε = {eps_ref} (≈ 8/255)", color=INK_MUTED, fontsize=9,
            transform=ax.get_xaxis_transform(), ha="left", va="top")
    ax.axhline(1 / 3, color=INK_MUTED, linestyle="-.", linewidth=1)
    ax.text(ax.get_xlim()[1], 1 / 3, "випадкове вгадування (33%) ", color=INK_MUTED, fontsize=9,
            ha="right", va="bottom")

    ax.set_ylim(0, 1.0)
    ax.set_xlim(left=0)
    ax.set_xlabel("Бюджет збурення ε (L∞, шкала [0, 1])")
    ax.set_ylabel("Точність на тестовій вибірці")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

    # Додаткова вісь зверху — той самий ε у звичних одиницях яскравості /255
    sec = ax.secondary_xaxis("top", functions=(lambda e: e * 255, lambda p: p / 255))
    sec.set_xlabel("ε × 255 (рівні яскравості)", color=INK_MUTED, fontsize=10)
    sec.tick_params(colors=INK_MUTED, labelsize=9)


curves_std = {
    ("Стандартна", "FGSM"): fgsm_results_df,
    ("Стандартна", "PGD"): pgd_results_df,
}

# ---------------------------------------------------------------
# Рис. 5. FGSM vs PGD на стандартній моделі (повний діапазон і збільшення)
# ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})

plot_robustness(ax1, curves_std)
ax1.set_title("Стійкість стандартної моделі: FGSM vs PGD", pad=34)
ax1.legend(loc="upper right", bbox_to_anchor=(1.0, 0.93))

# Праворуч — лише зона малих збурень, де відбувається «обвал» точності
zoom = {k: v[v["epsilon"] <= 0.05] for k, v in curves_std.items()}
plot_robustness(ax2, zoom, annotate=True)
ax2.set_title("Зона непомітних збурень (ε ≤ 0.05)", pad=34)

fig.tight_layout()
save_fig(fig, "fig05_fgsm_vs_pgd_standard")
plt.show()

display(style_summary(summarize(curves_std)))

**Інтерпретація.**

* Точність стандартної моделі **різко падає вже за ледь помітних збурень**: в авторському запуску при ε = 0.03 (≈ 8/255, тобто зміна яскравості пікселя лише на ~8 рівнів із 255) FGSM знижує її з 96.6 % до ≈ 41 %, а PGD — до ≈ 20 %, тобто **нижче рівня випадкового вгадування (33 %)**.
* **PGD стабільно сильніша за FGSM**: ітеративна атака знаходить змагальний приклад за меншого бюджету (ε₅₀ ≈ 0.020 проти ≈ 0.027 для FGSM). Тому оцінювати стійкість лише за FGSM недостатньо — це дає завищену, оптимістичну оцінку.
* За ε ≥ 0.1 обидві атаки зводять точність практично до нуля: модель повністю контрольована зловмисником.

## 9. Візуалізація змагальних прикладів

Для кожного прикладу показано:

1. **оригінал** з передбаченням моделі та її впевненістю;
2. **збурення** $\delta = x_{adv} - x$, лінійно підсилене для показу (сірий — без змін, світліше/темніше — зсув пікселя вгору/вниз);
3. **змагальне зображення** з новим (хибним) передбаченням;
4. **ймовірності класів** до і після атаки.

Відбираються лише приклади, які модель класифікувала **правильно** до атаки і **помилково** після неї (успішні атаки).

In [ ]:
def collect_adversarial_examples(model, loader, attack_fn, max_examples=3, only_successful=True):
    """Збирає пари (оригінал, змагальний приклад) разом із передбаченнями та ймовірностями."""
    model.eval()
    examples = []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        adv_images = attack_fn(images, labels)
        with torch.no_grad():
            clean_probs = F.softmax(model(images), dim=1)
            adv_probs = F.softmax(model(adv_images), dim=1)

        for i in range(images.size(0)):
            clean_pred, adv_pred, y = clean_probs[i].argmax().item(), adv_probs[i].argmax().item(), labels[i].item()
            if only_successful and not (clean_pred == y and adv_pred != y):
                continue
            examples.append({
                "original": images[i].cpu(), "adversarial": adv_images[i].cpu(),
                "label": y, "clean_pred": clean_pred, "adv_pred": adv_pred,
                "clean_probs": clean_probs[i].cpu().numpy(), "adv_probs": adv_probs[i].cpu().numpy(),
            })
            if len(examples) >= max_examples:
                return examples
    return examples


def show_adversarial_examples(examples, class_names, epsilon, title, fig_name=None):
    """Рядок на приклад: оригінал | карта збурення | змагальне зображення | ймовірності класів."""
    if not examples:
        print("Успішних змагальних прикладів не знайдено.")
        return

    n = len(examples)
    fig, axes = plt.subplots(n, 4, figsize=(13, 3.7 * n), squeeze=False,
                             gridspec_kw={"width_ratios": [1, 1, 1, 1.25]})
    c_clean, c_adv = MODEL_COLORS["Стандартна"], MODEL_COLORS["Змагальне навчання"]

    for r, ex in enumerate(examples):
        orig, adv = ex["original"], ex["adversarial"]
        delta = adv - orig
        # Збурення в межах [-eps, eps] лінійно розтягуємо на [0, 1]: сірий = 0, світле = +, темне = -
        delta_vis = (delta / (2 * max(epsilon, 1e-8)) + 0.5).clamp(0, 1)
        y = class_names[ex["label"]]
        cp, ap = class_names[ex["clean_pred"]], class_names[ex["adv_pred"]]

        axes[r][0].imshow(orig.permute(1, 2, 0).clamp(0, 1))
        axes[r][0].set_title(f"Оригінал\nсправжній: {y}\nпрогноз: {cp} ✓ ({ex['clean_probs'].max():.0%})", fontsize=10)

        axes[r][1].imshow(delta_vis.permute(1, 2, 0))
        axes[r][1].set_title(f"Збурення δ (підсилено ×{1 / (2 * epsilon):.0f})\nmax |δ| = {delta.abs().max():.3f}", fontsize=10)

        axes[r][2].imshow(adv.permute(1, 2, 0).clamp(0, 1))
        axes[r][2].set_title(f"Змагальний приклад\nсправжній: {y}\nпрогноз: {ap} ✗ ({ex['adv_probs'].max():.0%})", fontsize=10)

        for ax in axes[r][:3]:
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)

        # Ймовірності класів до/після атаки
        ax = axes[r][3]
        yk = np.arange(len(class_names))
        ax.barh(yk - 0.2, ex["clean_probs"], height=0.38, color=c_clean, label="до атаки")
        ax.barh(yk + 0.2, ex["adv_probs"], height=0.38, color=c_adv, hatch="////",
                edgecolor="white", linewidth=0, label="після атаки")
        ax.set_yticks(yk, class_names)
        ax.invert_yaxis()
        ax.set_xlim(0, 1)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.grid(axis="y", visible=False)
        ax.set_title("Ймовірності класів", fontsize=10)

    handles, legend_labels = axes[0][3].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="upper right", ncol=2, fontsize=9)
    fig.suptitle(title, fontweight="bold", x=0.02, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    if fig_name:
        save_fig(fig, fig_name)
    plt.show()

In [ ]:
# Рис. 6. FGSM, eps = 0.03
eps = 0.03
fgsm_examples = collect_adversarial_examples(
    model, test_loader, lambda x, y: generate_fgsm(model, x, y, criterion, eps), max_examples=3)
show_adversarial_examples(fgsm_examples, class_names, eps,
                          f"FGSM, ε = {eps} (≈ {eps * 255:.0f}/255): збурення непомітне, але прогноз хибний",
                          "fig06_examples_fgsm")

In [ ]:
# Рис. 7. PGD, eps = 0.02 — навіть менший бюджет, ніж у FGSM вище
eps = 0.02
set_seed()
pgd_examples = collect_adversarial_examples(
    model, test_loader, lambda x, y: pgd_attack(model, x, y, criterion, eps), max_examples=3)
show_adversarial_examples(pgd_examples, class_names, eps,
                          f"PGD, ε = {eps} (≈ {eps * 255:.0f}/255)", "fig07_examples_pgd")

In [ ]:
# Рис. 8. PGD, eps = 0.15 — великий бюджет: збурення вже помітне оком
eps = 0.15
set_seed()
pgd_examples_big = collect_adversarial_examples(
    model, test_loader, lambda x, y: pgd_attack(model, x, y, criterion, eps), max_examples=3)
show_adversarial_examples(pgd_examples_big, class_names, eps,
                          f"PGD, ε = {eps} (≈ {eps * 255:.0f}/255): шум стає видимим", "fig08_examples_pgd_large")

**Інтерпретація.** За ε ≤ 0.03 змагальне зображення візуально не відрізняється від оригіналу — людина впевнено розпізнає жест, тоді як модель помиляється, часто **з високою впевненістю**. Збурення розподілене по всьому кадру (включно з фоном), а не лише по руці: атака експлуатує не семантичні ознаки, а чутливість моделі до дрібних текстурних змін. За ε = 0.15 шум стає помітним, тож такий сценарій менш реалістичний для непомітної атаки.

## 10. Захист: змагальне навчання (adversarial training)

Ідея: під час навчання показувати моделі не лише чисті, а й **змагальні приклади з правильними мітками**. Формально це задача min-max:

$$\min_\theta \; \mathbb{E}_{(x,y)} \Big[\tfrac12 L(f_\theta(x), y) + \tfrac12 \max_{\|\delta\|_\infty \le \varepsilon} L(f_\theta(x+\delta), y)\Big]$$

Внутрішній максимум наближаємо однією ітерацією FGSM з $\varepsilon_{train} = 0.03$, яка генерується «на льоту» для поточного стану моделі в кожному батчі. Архітектура, оптимізатор і кількість епох — ті самі, що й у базової моделі, тому порівняння чесне.

In [ ]:
TRAIN_EPSILON = 0.03

set_seed()
adv_model = CNN().to(device)
adv_optimizer = torch.optim.Adam(adv_model.parameters(), lr=LR)

history_adv = {"loss": [], "train_acc": [], "train_adv_acc": [], "test_acc": []}

for epoch in range(NUM_EPOCHS):
    adv_model.train()
    running_loss, correct_clean, correct_adv, total = 0.0, 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # 1) генеруємо FGSM-приклади для поточних ваг моделі
        adv_images = generate_fgsm(adv_model, images, labels, criterion, TRAIN_EPSILON)

        # 2) втрати на чистих і змагальних прикладах, 50/50
        clean_outputs = adv_model(images)
        adv_outputs = adv_model(adv_images)
        loss = 0.5 * criterion(clean_outputs, labels) + 0.5 * criterion(adv_outputs, labels)

        # 3) звичайний крок оптимізації
        adv_optimizer.zero_grad()
        loss.backward()
        adv_optimizer.step()

        running_loss += loss.item()
        correct_clean += (clean_outputs.argmax(dim=1) == labels).sum().item()
        correct_adv += (adv_outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    history_adv["loss"].append(running_loss / len(train_loader))
    history_adv["train_acc"].append(correct_clean / total)
    history_adv["train_adv_acc"].append(correct_adv / total)
    history_adv["test_acc"].append(evaluate_clean(adv_model, test_loader))

    print(f"Epoch {epoch + 1:2d}/{NUM_EPOCHS} | loss {history_adv['loss'][-1]:.4f} | "
          f"train clean {history_adv['train_acc'][-1]:.4f} | train adv {history_adv['train_adv_acc'][-1]:.4f} | "
          f"test clean {history_adv['test_acc'][-1]:.4f}")

adv_clean_accuracy = history_adv["test_acc"][-1]
print(f"\nТочність моделі зі змагальним навчанням на чистих тестових даних: {adv_clean_accuracy:.4f}")

In [ ]:
# ---------------------------------------------------------------
# Рис. 9. Динаміка навчання: стандартна модель vs змагальне навчання
# ---------------------------------------------------------------
c_std, c_adv = MODEL_COLORS["Стандартна"], MODEL_COLORS["Змагальне навчання"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history_std["loss"], color=c_std, marker="o", label="Стандартна")
ax1.plot(epochs, history_adv["loss"], color=c_adv, marker="s", label="Змагальне навчання")
ax1.set_title("Функція втрат (train)")
ax1.set_xlabel("Епоха"); ax1.set_ylabel("Cross-entropy")
ax1.set_ylim(bottom=0)
ax1.legend()

ax2.plot(epochs, history_std["test_acc"], color=c_std, marker="o", label="Стандартна — test, чисті")
ax2.plot(epochs, history_adv["test_acc"], color=c_adv, marker="s", label="Змагальне навч. — test, чисті")
ax2.plot(epochs, history_adv["train_adv_acc"], color=c_adv, marker="s", linestyle="--",
         label=f"Змагальне навч. — train, FGSM ε={TRAIN_EPSILON}")
ax2.set_title("Точність")
ax2.set_xlabel("Епоха"); ax2.set_ylabel("Accuracy")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax2.legend(loc="lower right", fontsize=9)

for ax in (ax1, ax2):
    ax.set_xticks(epochs)

fig.tight_layout()
save_fig(fig, "fig09_training_curves_comparison")
plt.show()

In [ ]:
# Ширша сітка epsilon — захищена модель тримається довше, тож дивимося далі
EPSILONS_ADV = [0.000, 0.0025, 0.005, 0.0075, 0.010, 0.0125, 0.015, 0.0175, 0.020, 0.025,
                0.030, 0.035, 0.040, 0.045, 0.050, 0.055, 0.060, 0.065, 0.070, 0.075,
                0.080, 0.090, 0.100, 0.110, 0.120, 0.130, 0.140, 0.150, 0.175, 0.200,
                0.225, 0.250, 0.300]

adv_fgsm_results_df = run_sweep(lambda e: eval_fgsm(adv_model, test_loader, criterion, e), EPSILONS_ADV, "FGSM")

In [ ]:
set_seed()
adv_pgd_results_df = run_sweep(lambda e: evaluate_pgd(adv_model, test_loader, criterion, e), EPSILONS_ADV, "PGD ")

## 11. Стійкість до і після змагального навчання

Головний результат роботи. На рис. 10 колір позначає модель (синій — стандартна, помаранчевий — змагальне навчання), стиль лінії — атаку (суцільна — FGSM, штрихова — PGD). Рис. 11 показує ті ж дані для кількох характерних бюджетів ε у вигляді стовпчиків.

In [ ]:
curves_all = {
    ("Стандартна", "FGSM"): fgsm_results_df,
    ("Стандартна", "PGD"): pgd_results_df,
    ("Змагальне навчання", "FGSM"): adv_fgsm_results_df,
    ("Змагальне навчання", "PGD"): adv_pgd_results_df,
}

# ---------------------------------------------------------------
# Рис. 10. Криві стійкості всіх чотирьох конфігурацій
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 5.8))
plot_robustness(ax, curves_all, annotate=False)

# Зона, у якій навчався захист
ax.axvspan(0, TRAIN_EPSILON, color=MODEL_COLORS["Змагальне навчання"], alpha=0.07, linewidth=0)
ax.text(TRAIN_EPSILON / 2, 0.04, f"ε_train\n= {TRAIN_EPSILON}", ha="center", fontsize=9, color=INK_MUTED)

ax.set_title("Стійкість до змагальних атак до та після змагального навчання", pad=34)
ax.legend(loc="upper right", bbox_to_anchor=(1.0, 0.93))
fig.tight_layout()
save_fig(fig, "fig10_robustness_before_after")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Рис. 11. Точність за фіксованих бюджетів ε (групові стовпчики)
# ---------------------------------------------------------------
BAR_EPS = [0.01, 0.03, 0.05, 0.10]
configs = list(curves_all.keys())
width = 0.2
x = np.arange(len(BAR_EPS))

fig, ax = plt.subplots(figsize=(12, 5))
for k, (model_name, attack) in enumerate(configs):
    vals = [acc_at(curves_all[(model_name, attack)], e) for e in BAR_EPS]
    pos = x + (k - (len(configs) - 1) / 2) * width
    bars = ax.bar(pos, vals, width=width * 0.92, color=MODEL_COLORS[model_name],
                  hatch=ATTACK_HATCH[attack], edgecolor="white", linewidth=0,
                  label=f"{model_name} — {attack}")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.0%}", ha="center", va="bottom",
                fontsize=8.5, color=INK)

ax.axhline(1 / 3, color=INK_MUTED, linestyle="-.", linewidth=1)
ax.text(x[-1] + 0.5, 1 / 3, " випадкове\n вгадування", color=INK_MUTED, fontsize=9, va="center")
ax.set_xticks(x, [f"ε = {e}\n(≈ {e * 255:.1f}/255)" for e in BAR_EPS])
ax.set_ylim(0, 1.1)
ax.set_xlim(-0.5, len(BAR_EPS) - 0.1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylabel("Точність на тестовій вибірці")
ax.grid(axis="x", visible=False)
ax.set_title("Точність під атакою для характерних бюджетів збурення")
ax.legend(ncol=2, loc="upper right", fontsize=9)
fig.tight_layout()
save_fig(fig, "fig11_accuracy_bars")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Рис. 12. Одне зображення під PGD-атакою зі зростаючим ε:
# верхній рядок — стандартна модель, нижній — після змагального навчання
# ---------------------------------------------------------------
STRIP_EPS = [0.0, 0.01, 0.03, 0.05, 0.10, 0.20]

# Беремо перше тестове зображення, яке обидві моделі класифікують правильно
x_img, y_img = None, None
for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    with torch.no_grad():
        ok = (model(images).argmax(1) == labels) & (adv_model(images).argmax(1) == labels)
    if ok.any():
        i = ok.nonzero()[0].item()
        x_img, y_img = images[i:i + 1], labels[i:i + 1]
        break

strip_models = [("Стандартна", model), ("Змагальне навчання", adv_model)]
fig, axes = plt.subplots(len(strip_models), len(STRIP_EPS), figsize=(2.2 * len(STRIP_EPS), 5.4), squeeze=False)

for r, (name, m) in enumerate(strip_models):
    for c, e in enumerate(STRIP_EPS):
        set_seed()
        x_adv = pgd_attack(m, x_img, y_img, criterion, e) if e > 0 else x_img
        with torch.no_grad():
            probs = F.softmax(m(x_adv), dim=1)[0]
        pred = probs.argmax().item()
        mark = "✓" if pred == y_img.item() else "✗"

        ax = axes[r][c]
        ax.imshow(x_adv[0].cpu().permute(1, 2, 0).clamp(0, 1))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_xlabel(f"{class_names[pred]} {mark} ({probs[pred]:.0%})", fontsize=10, color=INK)
        if r == 0:
            ax.set_title(f"ε = {e}", fontsize=11)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(MODEL_COLORS[name])
            spine.set_linewidth(2.5)
    axes[r][0].set_ylabel(name, fontsize=11)

fig.suptitle(f"PGD-атака на одне зображення (справжній клас: {class_names[y_img.item()]})",
             fontweight="bold")
fig.tight_layout()
save_fig(fig, "fig12_epsilon_strip")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Табл. 1. Зведені метрики стійкості (для статті)
# ---------------------------------------------------------------
summary_table = summarize(curves_all)
summary_table.to_csv(FIG_DIR / "table1_robustness_summary.csv", index=False)
display(style_summary(summary_table))

In [ ]:
# ---------------------------------------------------------------
# Автоматичне формулювання ключових результатів за фактичними числами цього запуску
# ---------------------------------------------------------------
S = summary_table.set_index(["Модель", "Атака"])
std, adv = "Стандартна", "Змагальне навчання"


def e50(key):
    v = S.loc[key, "ε₅₀"]
    return f"> {max(curves_all[key]['epsilon'])}" if np.isnan(v) else f"≈ {v:.3f}"


gain_pgd = S.loc[(adv, "PGD"), "ε=0.03"] - S.loc[(std, "PGD"), "ε=0.03"]
clean_delta = S.loc[(adv, "FGSM"), "Clean"] - S.loc[(std, "FGSM"), "Clean"]

display(Markdown(f"""
### Ключові результати цього запуску

1. **Якість на чистих даних.** Стандартна CNN: **{pct(S.loc[(std, 'FGSM'), 'Clean'])}**, модель зі змагальним навчанням: **{pct(S.loc[(adv, 'FGSM'), 'Clean'])}** (різниця {100 * clean_delta:+.1f} п.п.).
2. **Вразливість стандартної моделі.** При ε = 0.03 (≈ 8/255) точність падає до **{pct(S.loc[(std, 'FGSM'), 'ε=0.03'])}** під FGSM і до **{pct(S.loc[(std, 'PGD'), 'ε=0.03'])}** під PGD. Поріг 50 % точності досягається вже при ε₅₀ {e50((std, 'FGSM'))} (FGSM) та {e50((std, 'PGD'))} (PGD).
3. **Ефект змагального навчання.** При ε = 0.03 точність захищеної моделі становить **{pct(S.loc[(adv, 'FGSM'), 'ε=0.03'])}** (FGSM) і **{pct(S.loc[(adv, 'PGD'), 'ε=0.03'])}** (PGD), тобто приріст стійкості до PGD — **{100 * gain_pgd:+.1f} п.п.** ε₅₀ зростає до {e50((adv, 'FGSM'))} (FGSM) та {e50((adv, 'PGD'))} (PGD).
4. **Інтегральна стійкість** (AUC на [0; 0.1]): стандартна модель — {S.loc[(std, 'FGSM'), 'AUC[0;0.1]']:.3f} / {S.loc[(std, 'PGD'), 'AUC[0;0.1]']:.3f} (FGSM / PGD), після захисту — {S.loc[(adv, 'FGSM'), 'AUC[0;0.1]']:.3f} / {S.loc[(adv, 'PGD'), 'AUC[0;0.1]']:.3f}.
"""))

## 12. Висновки (для публікації)

Числові значення нижче наведено за авторським запуском експерименту; актуальні числа для поточного запуску формуються автоматично в клітинці вище і в Табл. 1 (`figures/table1_robustness_summary.csv`).

**1. Висока точність не означає надійність.** Компактна CNN досягає ≈ 96.6 % точності на чистих тестових даних задачі Rock-Paper-Scissors, однак є вкрай вразливою до змагальних збурень. Зміна яскравості кожного пікселя лише на ~8 рівнів із 255 (ε = 0.03) — візуально непомітна для людини — знижує точність до ≈ 41 % (FGSM) і ≈ 20 % (PGD), тобто нижче рівня випадкового вгадування (33 %). За ε ≥ 0.1 модель помиляється майже на всіх прикладах.

**2. Ітеративні атаки суттєво сильніші за одноетапні.** PGD при тому самому бюджеті стабільно знижує точність сильніше, ніж FGSM (ε₅₀ ≈ 0.020 проти ≈ 0.027). Отже, оцінювання стійкості лише за FGSM дає завищену (оптимістичну) картину; для аудиту безпеки ML-моделей слід використовувати щонайменше PGD.

**3. Змагальне навчання ефективне і майже «безкоштовне».** Навчання на суміші чистих і FGSM-прикладів (ε_train = 0.03) зберегло точність на чистих даних (≈ 96.8 % проти 96.6 %) і підвищило точність під атакою при ε = 0.03 з ≈ 41 % до ≈ 92 % (FGSM) та з ≈ 20 % до ≈ 92 % (PGD). Бюджет, за якого точність падає нижче 50 % під FGSM, зріс приблизно вчетверо — з ≈ 0.027 до ≈ 0.10.

**4. Захист не є абсолютним.** Стійкість захищеної моделі поступово знижується зі зростанням ε за межі бюджету навчання, і за великих збурень (ε ≥ 0.2) атаки знову ефективні. Змагальне навчання зсуває межу вразливості, але не усуває її.

**Практичні рекомендації.** (1) Включати змагальне тестування (FGSM + PGD) у стандартну процедуру оцінки моделей комп'ютерного зору перед розгортанням; (2) звітувати не одне значення accuracy, а криву accuracy(ε) та інтегральні метрики (ε₅₀, AUC); (3) використовувати змагальне навчання як базовий захист, обираючи ε_train відповідно до реалістичної моделі загроз.

**Обмеження дослідження.**

* Розглянуто лише атаки *white-box* з обмеженням $L_\infty$; атаки *black-box*, фізичні (надруковані) збурення та інші норми ($L_2$, $L_0$) не досліджувалися.
* Модель невелика, зображення зменшені до 64×64, фон однорідний; для складніших наборів даних кількісні результати можуть відрізнятися.
* Захист навчався на одноетапних FGSM-прикладах. Такий підхід схильний до *gradient masking* і «катастрофічного перенавчання» (catastrophic overfitting), тому для остаточних висновків про стійкість варто перевірити модель сильнішими атаками (PGD із більшою кількістю кроків і випадкових рестартів, AutoAttack).
* У первинній версії експерименту PGD використовувала фіксований крок α = 0.005, через що за ε > 0.05 вона не могла повністю використати бюджет і виглядала слабшою за FGSM на захищеній моделі. У цій версії крок масштабується з ε (α = 2.5·ε/steps); значення для великих ε слід брати з поточного запуску.
* Результати отримано для одного зерна генератора та однієї тестової вибірки з 438 зображень (одне зображення ≈ 0.23 п.п. точності); для статистично строгих висновків доцільно повторити експеримент для кількох зерен і навести довірчі інтервали.

**Список літератури**

1. Goodfellow I. J., Shlens J., Szegedy C. *Explaining and Harnessing Adversarial Examples.* ICLR, 2015.
2. Madry A., Makelov A., Schmidt L., Tsipras D., Vladu A. *Towards Deep Learning Models Resistant to Adversarial Attacks.* ICLR, 2018.
3. Wong E., Rice L., Kolter J. Z. *Fast is Better than Free: Revisiting Adversarial Training.* ICLR, 2020.
4. Croce F., Hein M. *Reliable Evaluation of Adversarial Robustness with an Ensemble of Diverse Parameter-free Attacks (AutoAttack).* ICML, 2020.